# Climate indicators

This notebook reads the processed **simulated** dataset and recomputes the statistics the explorer shows for Spain and extreme heat.

The file `data/processed/climate.json` is labeled simulated. Nothing here is an official climate measurement.

Formulas live in `src/climate_pipeline/indicators.py` and are the same functions the API tests call.

In [ ]:
import json
from pathlib import Path

from climate_pipeline.indicators import (
    anomaly,
    baseline_mean,
    heat_risk_index,
    linear_trend,
    percent_change,
    period_mean,
)
from climate_pipeline.insights import baseline_sentence

ROOT = Path("..").resolve()
payload = json.loads((ROOT / "data" / "processed" / "climate.json").read_text(encoding="utf-8"))
print(payload["meta"]["label"])
print(payload["meta"]["notes"][0])

In [ ]:
years = payload["years"]
spain = payload["series"]["ESP"]
heat = spain["heat_days"]
temperature = spain["temperature"]

baseline = baseline_mean(years, heat, 1980, 2000)
recent = period_mean(years, heat, 2005, 2025)
temp_base = baseline_mean(years, temperature, 1980, 2000)
temp_recent = period_mean(years, temperature, 2005, 2025)
window = [(year, value) for year, value in zip(years, heat) if 2005 <= year <= 2025]
trend = linear_trend([year for year, _ in window], [value for _, value in window])
change = percent_change(recent, baseline)

print(f"Heat days, 1980–2000: {baseline:.1f} days/year")
print(f"Heat days, 2005–2025: {recent:.1f} days/year")
print(f"Change: {change:+.0f}%")
print(f"Heat risk index, recent mean: {heat_risk_index(recent):.0f} / 100")
print(f"Temperature anomaly, recent vs baseline: {anomaly(temp_recent, temp_base):+.2f} °C")
print(f"OLS slope, 2005–2025: {trend['slope'] * 10:.1f} days/decade")
print(baseline_sentence("Spain", "Extreme heat days", change, recent, baseline, "days", 1980, 2000))

## Missing data stays missing

Greenland drops every year divisible by 7. Those years are null. `baseline_mean` skips them and returns null when fewer than five points remain. It does not fill the gap with zero.

In [ ]:
greenland = payload["series"]["GRL"]
missing_years = [year for year, flag in zip(years, greenland["coverage"]) if not flag]
print("Greenland missing years:", missing_years)
print("2023 temperature is null:", greenland["temperature"][years.index(2023)] is None)

short = baseline_mean([1980, 1981, 1982], [1.0, None, 2.0], 1980, 1982)
print("Baseline with only two points:", short)